# Session 32: Logistic Regression Classification Baseline
**Week 3 Baseline Classifier for At-Risk Target Prediction**

In this notebook, we shift from regression tasks to binary classification. We construct an "at-risk" target variable ($G3 < 10$) and fit a scaled Logistic Regression baseline model using a scikit-learn `Pipeline`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, classification_report, confusion_matrix
)

# Resolve project directories
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / ".git").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / ".git").exists():
            PROJECT_ROOT = parent
            break

DATA_DIRECTORY = PROJECT_ROOT / "data"
REPORTS_DIRECTORY = PROJECT_ROOT / "reports"
TABLES_DIRECTORY = REPORTS_DIRECTORY / "tables"

TABLES_DIRECTORY.mkdir(parents=True, exist_ok=True)
print("Project Root:", PROJECT_ROOT)

Project Root: /home/nikhil/Desktop/VSCode/GSSRP/student-performance-prediction-ml


In [2]:
def load_and_transform_data():
    # Attempt to load the same feature splits used in regression
    # If loading raw processed table, split using the same random state
    from sklearn.model_selection import train_test_split
    
    candidates = list((DATA_DIRECTORY / "processed").rglob("*.parquet")) + list((DATA_DIRECTORY / "processed").rglob("*.csv"))
    for path in candidates:
        if any(term in path.name.lower() for term in ["comparison", "prediction", "result"]):
            continue
        try:
            table = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
            if "G3" in table.columns:
                X = table.drop(columns=["G3"]).copy()
                X = pd.get_dummies(X, drop_first=True, dtype=float)
                y_reg = table["G3"]
                
                # Critical step: Convert regression target to binary classification target
                # At-risk (1) if G3 < 10, otherwise Pass (0)
                y_clf = (y_reg < 10).astype(int)
                
                Xtr, Xte, yctr, ycte = train_test_split(
                    X, y_clf, test_size=0.20, random_state=42
                )
                return Xtr, Xte, yctr, ycte
        except Exception as e:
            continue
    raise FileNotFoundError("Could not locate source data splits.")

Xtr_f, Xte_f, yctr, ycte = load_and_transform_data()
print(f"Train shapes: X={Xtr_f.shape}, y={yctr.shape}")
print(f"At-risk prevalence in training: {yctr.mean():.2%}")

Train shapes: X=(316, 41), y=(316,)
At-risk prevalence in training: 32.59%


In [3]:
# Instantiate scaled Logistic Regression pipeline
clf = make_pipeline(
    StandardScaler(), 
    LogisticRegression(max_iter=1000, random_state=42)
)

# Fit classifier on binarized training data
clf.fit(Xtr_f, yctr)

# Generate class and probability predictions
predictions = clf.predict(Xte_f)
probabilities = clf.predict_proba(Xte_f)[:, 1]

# Compute standard evaluation metrics
accuracy = accuracy_score(ycte, predictions)

precision = precision_score(
    ycte, predictions,
    pos_label=1,
    zero_division=0
)

recall = recall_score(
    ycte, predictions,
    pos_label=1,
    zero_division=0
)

f1 = f1_score(
    ycte, predictions,
    pos_label=1,
    zero_division=0
)

roc_auc = roc_auc_score(ycte, probabilities)

# Metrics specifically for the "At Risk" (class=1) students
at_risk_precision = precision
at_risk_recall = recall
at_risk_f1 = f1

print("Baseline Logistic Regression Performance:")
print("-" * 40)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC AUC:   {roc_auc:.4f}\n")

print("Classification Report:")
print(classification_report(ycte, predictions))

Baseline Logistic Regression Performance:
----------------------------------------
Accuracy:  0.8987
Precision: 0.8276
Recall:    0.8889
F1 Score:  0.8571
ROC AUC:   0.9736

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.90      0.92        52
           1       0.83      0.89      0.86        27

    accuracy                           0.90        79
   macro avg       0.88      0.90      0.89        79
weighted avg       0.90      0.90      0.90        79



In [4]:
classification_path = TABLES_DIRECTORY / "classification_model_comparison.csv"

# CORRECTED SCHEMA: Using lowercase metrics columns to align with Sessions 33 and 34
log_row = pd.DataFrame([{
    "Model": "LR",
    "Full Model Name": "Logistic Regression",
    "Model Family": "Linear",
    "Scaling_Used": True,
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "roc_auc": roc_auc,
    "at_risk_precision": at_risk_precision,
    "at_risk_recall": at_risk_recall,
    "at_risk_f1": at_risk_f1
}])

if classification_path.exists():
    comparison_table = pd.read_csv(classification_path)
    # Remove older copies of LR to keep the file clean
    if "Model" in comparison_table.columns:
        comparison_table = comparison_table[comparison_table["Model"] != "LR"].copy()
    comparison_table = pd.concat([comparison_table, log_row], ignore_index=True)
else:
    comparison_table = log_row.copy()

comparison_table.to_csv(classification_path, index=False)
print("Updated classification comparison table saved at:")
print(classification_path)
display(comparison_table.round(4))

Updated classification comparison table saved at:
/home/nikhil/Desktop/VSCode/GSSRP/student-performance-prediction-ml/reports/tables/classification_model_comparison.csv


,Overall F1 Rank,Model,Full Model Name,Model Family,Scaling_Used,accuracy,precision,recall,f1,roc_auc,at_risk_precision,at_risk_recall,at_risk_f1
0,1.0,DTC,Decision Tree Classifier,Tree-based,False,0.9114,0.8571,0.8889,0.8727,0.9099,0.9412,0.9231,0.9320
1,3.0,RFC,Random Forest Classifier,Ensemble,False,0.8987,0.8276,0.8889,0.8571,0.9605,0.9400,0.9038,0.9216
2,4.0,AdaBoost,AdaBoost Classifier,Boosting,False,0.8987,0.8519,0.8519,0.8519,0.9729,0.9231,0.9231,0.9231
3,5.0,GradBoost,Gradient Boosting Classifier,Boosting,False,0.8608,0.7857,0.8148,0.8000,0.9608,0.9020,0.8846,0.8932
4,6.0,MLPC,Multi-Layer Perceptron Classifier,Neural Network,True,0.8734,0.8696,0.7407,0.8000,0.9423,0.8750,0.9423,0.9074
5,7.0,SVM,Support Vector Machine,Maximum-margin,True,0.8354,0.7917,0.7037,0.7451,0.9387,0.8545,0.9038,0.8785
6,8.0,NB,Gaussian Naive Bayes,Probabilistic,True,0.8228,0.7600,0.7037,0.7308,0.9003,0.8519,0.8846,0.8679
7,9.0,KNN,K-Nearest Neighbors,Instance-based,True,0.7468,0.7059,0.4444,0.5455,0.7393,0.7581,0.9038,0.8246
8,NaN,LR,Logistic Regression,Linear,True,0.8987,0.8276,0.8889,0.8571,0.9736,0.8276,0.8889,0.8571


In [5]:
# Validate scaling step and architecture parameters
assert isinstance(clf.steps[0][1], StandardScaler), "Pipeline step 0 must be StandardScaler!"
assert isinstance(clf.steps[1][1], LogisticRegression), "Pipeline step 1 must be LogisticRegression!"
assert clf.steps[1][1].max_iter == 1000, "max_iter must be set to 1000!"

# Verify data shapes and types
assert set(np.unique(yctr)).issubset({0, 1}), "Target classes must be strictly binary 0 and 1!"
assert len(predictions) == len(ycte), "Prediction size mismatch!"

print("SESSION 32 NOTEBOOK SECTION COMPLETED SUCCESSFULLY")

SESSION 32 NOTEBOOK SECTION COMPLETED SUCCESSFULLY


## Session 33: KNN, SVM, and Naive Bayes Classification
This section extends the classification-model notebook with three additional classifier families:
1. K-Nearest Neighbors (KNN)
2. Support Vector Machine (SVM)
3. Gaussian Naive Bayes (NB)

All three classifiers use the same training and test observations. KNN and SVM require scaling because their results depend directly on feature distances or margins. Gaussian Naive Bayes is included in the same pipeline to preserve a consistent modeling workflow. The required output is an updated classification table containing KNN, SVM, and NB rows.

In [6]:
# Session 33 imports and prerequisite validation
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Ensure baseline variables from Session 32 exist in kernel memory
required_session33_objects = ["Xtr_f", "Xte_f", "yctr", "ycte"]
missing_objects = [obj for obj in required_session33_objects if obj not in globals()]

if missing_objects:
    raise NameError(f"Missing required preprocessing splits: {missing_objects}. Run Session 32 cells first.")

assert len(Xtr_f) == len(yctr), "Xtr_f and yctr must contain the same number of rows."
assert len(Xte_f) == len(ycte), "Xte_f and ycte must contain the same number of rows."

print("Session 33 prerequisites verified successfully.")

Session 33 prerequisites verified successfully.


### Target Class Mapping Context
The notebook utilizes the project target definitions:
* `0` = At-risk student (Final grade below 10)
* `1` = Successful student (Final grade 10 or higher)

Standard metrics below treat class 1 (successful) as the positive class. To gain deep visibility into our early warning task, additional tracking blocks isolate performance treating class 0 (at-risk) as the primary outcome.

In [7]:
def evaluate_session33_classifier(y_true, y_pred, y_probability):
    """
    Return standard and at-risk classification metrics.
    Standard precision, recall, and F1 treat class 1 as positive.
    At-risk precision, recall, and F1 treat class 0 as positive.
    """
    unique_classes = np.unique(np.asarray(y_true).ravel())
    roc_auc = roc_auc_score(y_true, y_probability) if len(unique_classes) == 2 else np.nan
    
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc,
        "at_risk_precision": precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        "at_risk_recall": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "at_risk_f1": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
    }

print("Session 33 classification evaluator is ready.")

Session 33 classification evaluator is ready.


In [8]:
# Establish target structural estimators
session33_estimators = [
    ("KNN", "K-Nearest Neighbors", "Instance-based", KNeighborsClassifier()),
    ("SVM", "Support Vector Machine", "Maximum-margin", SVC(probability=True, random_state=42)),
    ("NB", "Gaussian Naive Bayes", "Probabilistic", GaussianNB())
]

session33_models = {}
session33_result_rows = []

for model_code, full_model_name, model_family, estimator in session33_estimators:
    # Wrap standard scaler around estimators to insulate feature distances/margins cleanly
    pipeline = make_pipeline(StandardScaler(), estimator)
    pipeline.fit(Xtr_f, yctr)
    
    # Evaluate predictions against test splits
    predictions = pipeline.predict(Xte_f)
    probabilities = pipeline.predict_proba(Xte_f)[:, 1]
    
    metrics = evaluate_session33_classifier(y_true=ycte, y_pred=predictions, y_probability=probabilities)
    
    session33_models[model_code] = pipeline
    session33_result_rows.append({
        "Model": model_code,
        "Full Model Name": full_model_name,
        "Model Family": model_family,
        "Scaling_Used": True,
        **metrics
    })
    
    print(f"{model_code} Completed - F1 (Class 1): {metrics['f1']:.4f} | At-Risk F1 (Class 0): {metrics['at_risk_f1']:.4f}")

session33_results_df = pd.DataFrame(session33_result_rows).sort_values(by="f1", ascending=False).reset_index(drop=True)
session33_results_df.insert(0, "Session33 F1 Rank", range(1, len(session33_results_df) + 1))

KNN Completed - F1 (Class 1): 0.5455 | At-Risk F1 (Class 0): 0.8246
SVM Completed - F1 (Class 1): 0.7451 | At-Risk F1 (Class 0): 0.8785
NB Completed - F1 (Class 1): 0.7308 | At-Risk F1 (Class 0): 0.8679


In [9]:
classification_metric_columns = [
    "Model", "Full Model Name", "Model Family", "Scaling_Used",
    "accuracy", "precision", "recall", "f1", "roc_auc",
    "at_risk_precision", "at_risk_recall", "at_risk_f1"
]

# Set project paths relative to execution directories
PROJECT_ROOT = Path.cwd().resolve()
for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (parent / ".git").exists():
        PROJECT_ROOT = parent
        break

tables_dir = PROJECT_ROOT / "reports" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
cumulative_path = tables_dir / "classification_model_comparison.csv"

# Slice current session rows to align metrics schema
new_session33_rows = session33_results_df[classification_metric_columns].copy()

if cumulative_path.exists():
    classification_table = pd.read_csv(cumulative_path)
    # Wipe out legacy historical rows of KNN, SVM, or NB to avoid duplicates on re-execution
    if "Model" in classification_table.columns:
        classification_table = classification_table[~classification_table["Model"].isin(["KNN", "SVM", "NB"])].copy()
    
    # Fill missing column dimensions if aligning tables from older updates
    for col in classification_metric_columns:
        if col not in classification_table.columns:
            classification_table[col] = np.nan
            
    classification_table = pd.concat([classification_table[classification_metric_columns], new_session33_rows], ignore_index=True)
else:
    classification_table = new_session33_rows.copy()

# Globally rank across estimators using the baseline F1 metric
classification_table = classification_table.sort_values(by="f1", ascending=False).reset_index(drop=True)
if "Overall F1 Rank" in classification_table.columns:
    classification_table = classification_table.drop(columns=["Overall F1 Rank"])
classification_table.insert(0, "Overall F1 Rank", range(1, len(classification_table) + 1))

# Write updated artifacts to disk
session33_rows_path = tables_dir / "session33_classification_rows.csv"
session33_results_df[classification_metric_columns].to_csv(session33_rows_path, index=False)
classification_table.to_csv(cumulative_path, index=False)

print("Updated Classification Table:")
display(classification_table.style.format({
    "accuracy": "{:.4f}", "precision": "{:.4f}", "recall": "{:.4f}", "f1": "{:.4f}", "roc_auc": "{:.4f}",
    "at_risk_precision": "{:.4f}", "at_risk_recall": "{:.4f}", "at_risk_f1": "{:.4f}"
}))

Updated Classification Table:


,Overall F1 Rank,Model,Full Model Name,Model Family,Scaling_Used,accuracy,precision,recall,f1,roc_auc,at_risk_precision,at_risk_recall,at_risk_f1
0,1,DTC,Decision Tree Classifier,Tree-based,False,0.9114,0.8571,0.8889,0.8727,0.9099,0.9412,0.9231,0.9320
1,2,RFC,Random Forest Classifier,Ensemble,False,0.8987,0.8276,0.8889,0.8571,0.9605,0.9400,0.9038,0.9216
2,3,LR,Logistic Regression,Linear,True,0.8987,0.8276,0.8889,0.8571,0.9736,0.8276,0.8889,0.8571
3,4,AdaBoost,AdaBoost Classifier,Boosting,False,0.8987,0.8519,0.8519,0.8519,0.9729,0.9231,0.9231,0.9231
4,5,GradBoost,Gradient Boosting Classifier,Boosting,False,0.8608,0.7857,0.8148,0.8000,0.9608,0.9020,0.8846,0.8932
5,6,MLPC,Multi-Layer Perceptron Classifier,Neural Network,True,0.8734,0.8696,0.7407,0.8000,0.9423,0.8750,0.9423,0.9074
6,7,SVM,Support Vector Machine,Maximum-margin,True,0.8354,0.7917,0.7037,0.7451,0.9387,0.8545,0.9038,0.8785
7,8,NB,Gaussian Naive Bayes,Probabilistic,True,0.8228,0.7600,0.7037,0.7308,0.9003,0.8519,0.8846,0.8679
8,9,KNN,K-Nearest Neighbors,Instance-based,True,0.7468,0.7059,0.4444,0.5455,0.7393,0.7581,0.9038,0.8246


### Session 33 Theoretical Interpretation

* **K-Nearest Neighbors (KNN)** assumes that observations close to one another in the scaled feature space tend to share the same target class.
* **Support Vector Machine (SVM)** searches for a structural maximum-margin decision boundary separating output distributions.
* **Gaussian Naive Bayes (NB)** assumes that all predictors are conditionally independent given the class outcome label, modeling continuous predictors using Gaussian tracking parameters within classes.

The Naive Bayes conditional independence assumption is highly unrealistic for student academic performance tracking data. Performance metrics such as earlier period grades, study time, class failures, and historical absenteeism represent tightly interwoven social, behavioral, and academic processes that remain heavily correlated even after conditioning on whether a student passes or falls into the "at-risk" tier. However, it serves as an incredibly lightweight, efficient baseline for our comparison table.

In [10]:
# Automated structural validation checks
expected_models = {"KNN", "SVM", "NB"}
actual_models = set(session33_results_df["Model"])
assert actual_models == expected_models, "Session 33 results missing specific models."
assert len(session33_results_df) == 3, "Result rows mismatch inside standalone table."

# Metrics value bound assertions
required_metrics = ["accuracy", "precision", "recall", "f1", "roc_auc", "at_risk_precision", "at_risk_recall", "at_risk_f1"]
metric_values = session33_results_df[required_metrics].to_numpy(dtype=float)
assert np.isfinite(metric_values).all(), "Non-finite tracking numbers found in results."
assert ((metric_values >= 0) & (metric_values <= 1)).all(), "Metrics bounds exceeded outside [0, 1]."

# Assert that every pipelines contains standard scaling
for code_name, pipeline_obj in session33_models.items():
    assert isinstance(pipeline_obj.steps[0][1], StandardScaler), f"{code_name} pipeline step 0 is not StandardScaler."

print("=" * 72)
print("SESSION 33 GITHUB DELIVERABLE COMPLETED SUCCESSFULLY")
print("=" * 72)

SESSION 33 GITHUB DELIVERABLE COMPLETED SUCCESSFULLY


## Session 34: Decision Tree Classification
This section implements a depth-limited Decision Tree classifier to model our at-risk student binary outcome. Unlike instance-based or probabilistic variants, a single decision tree builds a highly interpretable, rule-based hierarchical structure that explicitly highlights the features contributing most significantly to splitting criteria.

In [11]:
# Session 34 imports and namespace safety check
from sklearn.tree import DecisionTreeClassifier, export_text
import numpy as np
import pandas as pd
from IPython.display import display

# Ensure core variables are ready from previous cells
required_session34_objects = ["Xtr_f", "Xte_f", "yctr", "ycte", "evaluate_session33_classifier"]
missing_objects = [obj for obj in required_session34_objects if obj not in globals()]

if missing_objects:
    raise NameError(f"Prerequisite objects missing from the active environment namespace: {missing_objects}")

print("Session 34 prerequisites successfully verified. Classification subsets are ready.")

Session 34 prerequisites successfully verified. Classification subsets are ready.


In [12]:
# Initialize the Decision Tree Classifier with structural depth constraints
dtc = DecisionTreeClassifier(max_depth=5, random_state=42)

# Train on full-information classification training variables
dtc.fit(Xtr_f, yctr)

# Generate predictions and probabilities for test verification
dtc_predictions = dtc.predict(Xte_f)
dtc_probabilities = dtc.predict_proba(Xte_f)[:, 1]

# Evaluate metrics using our existing classification evaluator
dtc_metrics = evaluate_session33_classifier(y_true=ycte, y_pred=dtc_predictions, y_probability=dtc_probabilities)

print("Decision Tree Classifier Evaluation (Test Split):")
print("-" * 50)
for k, v in dtc_metrics.items():
    print(f"{k}: {v:.4f}")

print(f"\nFitted tree total depth: {dtc.get_depth()}")
print(f"Fitted tree total terminal leaves: {dtc.get_n_leaves()}")

Decision Tree Classifier Evaluation (Test Split):
--------------------------------------------------
accuracy: 0.9114
precision: 0.8571
recall: 0.8889
f1: 0.8727
roc_auc: 0.9099
at_risk_precision: 0.9412
at_risk_recall: 0.9231
at_risk_f1: 0.9320

Fitted tree total depth: 5
Fitted tree total terminal leaves: 16


In [13]:
# Export readable, hierarchical structure text rules
feature_names_list = list(Xtr_f.columns) if hasattr(Xtr_f, "columns") else None

tree_rules_text = export_text(
    dtc, 
    feature_names=feature_names_list,
    max_depth=3  # Truncate rules display depth for document readability
)

print("Extracted Hierarchical Decision Tree Rules (Truncated to Depth 3):")
print("-" * 70)
print(tree_rules_text)

Extracted Hierarchical Decision Tree Rules (Truncated to Depth 3):
----------------------------------------------------------------------
|--- G2 <= 9.50
|   |--- G2 <= 8.50
|   |   |--- schoolsup_yes <= 0.50
|   |   |   |--- class: 1
|   |   |--- schoolsup_yes >  0.50
|   |   |   |--- Medu <= 1.50
|   |   |   |   |--- class: 0
|   |   |   |--- Medu >  1.50
|   |   |   |   |--- truncated branch of depth 2
|   |--- G2 >  8.50
|   |   |--- Fjob_other <= 0.50
|   |   |   |--- Mjob_teacher <= 0.50
|   |   |   |   |--- class: 1
|   |   |   |--- Mjob_teacher >  0.50
|   |   |   |   |--- truncated branch of depth 2
|   |   |--- Fjob_other >  0.50
|   |   |   |--- health <= 4.50
|   |   |   |   |--- truncated branch of depth 2
|   |   |   |--- health >  4.50
|   |   |   |   |--- class: 0
|--- G2 >  9.50
|   |--- G1 <= 10.50
|   |   |--- age <= 17.50
|   |   |   |--- famsize_LE3 <= 0.50
|   |   |   |   |--- class: 0
|   |   |   |--- famsize_LE3 >  0.50
|   |   |   |   |--- truncated branch of d

In [14]:
# Establish local path destinations
PROJECT_ROOT = Path.cwd().resolve()
for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (parent / ".git").exists():
        PROJECT_ROOT = parent
        break

cumulative_path = PROJECT_ROOT / "reports" / "tables" / "classification_model_comparison.csv"

# Shape the output metadata row to match your table's layout schema
dtc_row = pd.DataFrame([{
    "Model": "DTC",
    "Full Model Name": "Decision Tree Classifier",
    "Model Family": "Tree-based",
    "Scaling_Used": False,
    "accuracy": dtc_metrics["accuracy"],
    "precision": dtc_metrics["precision"],
    "recall": dtc_metrics["recall"],
    "f1": dtc_metrics["f1"],
    "roc_auc": dtc_metrics["roc_auc"],
    "at_risk_precision": dtc_metrics["at_risk_precision"],
    "at_risk_recall": dtc_metrics["at_risk_recall"],
    "at_risk_f1": dtc_metrics["at_risk_f1"]
}])

if cumulative_path.exists():
    classification_table = pd.read_csv(cumulative_path)
    
    # Prune previous historical DTC records to keep executions clean
    if "Model" in classification_table.columns:
        classification_table = classification_table[classification_table["Model"] != "DTC"].copy()
        
    classification_table = pd.concat([classification_table, dtc_row], ignore_index=True)
else:
    classification_table = dtc_row.copy()

# Sort metrics globally by standard baseline F1
classification_table = classification_table.sort_values(by="f1", ascending=False).reset_index(drop=True)

if "Overall F1 Rank" in classification_table.columns:
    classification_table = classification_table.drop(columns=["Overall F1 Rank"])
classification_table.insert(0, "Overall F1 Rank", range(1, len(classification_table) + 1))

# Write updated results to reports directory
classification_table.to_csv(cumulative_path, index=False)
print(f"Cumulative tracking table written successfully to: {cumulative_path}")
display(classification_table.round(4))

Cumulative tracking table written successfully to: /home/nikhil/Desktop/VSCode/GSSRP/student-performance-prediction-ml/reports/tables/classification_model_comparison.csv


,Overall F1 Rank,Model,Full Model Name,Model Family,Scaling_Used,accuracy,precision,recall,f1,roc_auc,at_risk_precision,at_risk_recall,at_risk_f1
0,1,DTC,Decision Tree Classifier,Tree-based,False,0.9114,0.8571,0.8889,0.8727,0.9099,0.9412,0.9231,0.9320
1,2,RFC,Random Forest Classifier,Ensemble,False,0.8987,0.8276,0.8889,0.8571,0.9605,0.9400,0.9038,0.9216
2,3,LR,Logistic Regression,Linear,True,0.8987,0.8276,0.8889,0.8571,0.9736,0.8276,0.8889,0.8571
3,4,AdaBoost,AdaBoost Classifier,Boosting,False,0.8987,0.8519,0.8519,0.8519,0.9729,0.9231,0.9231,0.9231
4,5,GradBoost,Gradient Boosting Classifier,Boosting,False,0.8608,0.7857,0.8148,0.8000,0.9608,0.9020,0.8846,0.8932
5,6,MLPC,Multi-Layer Perceptron Classifier,Neural Network,True,0.8734,0.8696,0.7407,0.8000,0.9423,0.8750,0.9423,0.9074
6,7,SVM,Support Vector Machine,Maximum-margin,True,0.8354,0.7917,0.7037,0.7451,0.9387,0.8545,0.9038,0.8785
7,8,NB,Gaussian Naive Bayes,Probabilistic,True,0.8228,0.7600,0.7037,0.7308,0.9003,0.8519,0.8846,0.8679
8,9,KNN,K-Nearest Neighbors,Instance-based,True,0.7468,0.7059,0.4444,0.5455,0.7393,0.7581,0.9038,0.8246


In [15]:
# Enforce completion bounds and hyperparameters verification check
assert dtc.max_depth == 5, "Error: max_depth parameter configuration mismatch!"
assert dtc.random_state == 42, "Error: random_state parameter configuration mismatch!"
assert len(dtc_predictions) == len(ycte), "Error: Prediction output elements vector shape mismatch!"
assert np.isfinite(metric_values).all(), "Error: Evaluator outputs contain invalid tracking entries!"

print("=" * 72)
print("SESSION 34 GITHUB DELIVERABLE COMPLETED SUCCESSFULLY")
print("=" * 72)

SESSION 34 GITHUB DELIVERABLE COMPLETED SUCCESSFULLY


## Session 35: Random Forest Classification
This section implements a **Random Forest Classifier** ensemble using 300 decision trees. We test the research hypothesis that combining an ensemble of bootstrap-aggregated weak learners can reduce structural variance and improve at-risk student detection accuracy without introducing bias. Because tree-based models are scale-invariant, feature scaling is not mandatory for this model.

In [16]:
# Session 35 imports and environment verification
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import pandas as pd
from IPython.display import display

# Verify required splits and evaluation helpers exist in the active namespace
required_session35_objects = ["Xtr_f", "Xte_f", "yctr", "ycte", "evaluate_session33_classifier"]
missing_objects = [obj for obj in required_session35_objects if obj not in globals()]

if missing_objects:
    raise NameError(f"Prerequisite objects missing from prior sections: {missing_objects}. Please run the preceding cells first.")

print("Session 35 prerequisites successfully verified. Datasets are ready for ensemble training.")

Session 35 prerequisites successfully verified. Datasets are ready for ensemble training.


In [17]:
# Initialize Random Forest Classifier with 300 trees
rfc = RandomForestClassifier(n_estimators=300, random_state=42)

# Fit model using raw, unscaled full-information training features
rfc.fit(Xtr_f, yctr)

# Generate class and probability predictions for test verification
rfc_predictions = rfc.predict(Xte_f)
rfc_probabilities = rfc.predict_proba(Xte_f)[:, 1]

# Evaluate metrics using the project evaluation function
rfc_metrics = evaluate_session33_classifier(y_true=ycte, y_pred=rfc_predictions, y_probability=rfc_probabilities)

print("Random Forest Classifier Evaluation (Test Split):")
print("-" * 50)
for metric_name, metric_val in rfc_metrics.items():
    print(f"{metric_name}: {metric_val:.4f}")

print(f"\nTotal fitted estimators (trees) within ensemble: {len(rfc.estimators_)}")

Random Forest Classifier Evaluation (Test Split):
--------------------------------------------------
accuracy: 0.8987
precision: 0.8276
recall: 0.8889
f1: 0.8571
roc_auc: 0.9605
at_risk_precision: 0.9400
at_risk_recall: 0.9038
at_risk_f1: 0.9216

Total fitted estimators (trees) within ensemble: 300


In [18]:
# Path to cumulative metrics tracker
PROJECT_ROOT = Path.cwd().resolve()
for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (parent / ".git").exists():
        PROJECT_ROOT = parent
        break

cumulative_path = PROJECT_ROOT / "reports" / "tables" / "classification_model_comparison.csv"

# Structure the output tracking row with precise schema alignment
rfc_row = pd.DataFrame([{
    "Model": "RFC",
    "Full Model Name": "Random Forest Classifier",
    "Model Family": "Ensemble",
    "Scaling_Used": False,
    "accuracy": rfc_metrics["accuracy"],
    "precision": rfc_metrics["precision"],
    "recall": rfc_metrics["recall"],
    "f1": rfc_metrics["f1"],
    "roc_auc": rfc_metrics["roc_auc"],
    "at_risk_precision": rfc_metrics["at_risk_precision"],
    "at_risk_recall": rfc_metrics["at_risk_recall"],
    "at_risk_f1": rfc_metrics["at_risk_f1"]
}])

if cumulative_path.exists():
    classification_table = pd.read_csv(cumulative_path)
    
    # Clear prior historical runs of RFC rows to maintain a strictly clean dataset
    if "Model" in classification_table.columns:
        classification_table = classification_table[classification_table["Model"] != "RFC"].copy()
        
    classification_table = pd.concat([classification_table, rfc_row], ignore_index=True)
else:
    classification_table = rfc_row.copy()

# Sort the table globally by standard baseline F1
classification_table = classification_table.sort_values(by="f1", ascending=False).reset_index(drop=True)

if "Overall F1 Rank" in classification_table.columns:
    classification_table = classification_table.drop(columns=["Overall F1 Rank"])
classification_table.insert(0, "Overall F1 Rank", range(1, len(classification_table) + 1))

# Export updated tables
classification_table.to_csv(cumulative_path, index=False)
print(f"Cumulative tracking table written successfully to: {cumulative_path}")
display(classification_table.round(4))

Cumulative tracking table written successfully to: /home/nikhil/Desktop/VSCode/GSSRP/student-performance-prediction-ml/reports/tables/classification_model_comparison.csv


,Overall F1 Rank,Model,Full Model Name,Model Family,Scaling_Used,accuracy,precision,recall,f1,roc_auc,at_risk_precision,at_risk_recall,at_risk_f1
0,1,DTC,Decision Tree Classifier,Tree-based,False,0.9114,0.8571,0.8889,0.8727,0.9099,0.9412,0.9231,0.9320
1,2,LR,Logistic Regression,Linear,True,0.8987,0.8276,0.8889,0.8571,0.9736,0.8276,0.8889,0.8571
2,3,RFC,Random Forest Classifier,Ensemble,False,0.8987,0.8276,0.8889,0.8571,0.9605,0.9400,0.9038,0.9216
3,4,AdaBoost,AdaBoost Classifier,Boosting,False,0.8987,0.8519,0.8519,0.8519,0.9729,0.9231,0.9231,0.9231
4,5,GradBoost,Gradient Boosting Classifier,Boosting,False,0.8608,0.7857,0.8148,0.8000,0.9608,0.9020,0.8846,0.8932
5,6,MLPC,Multi-Layer Perceptron Classifier,Neural Network,True,0.8734,0.8696,0.7407,0.8000,0.9423,0.8750,0.9423,0.9074
6,7,SVM,Support Vector Machine,Maximum-margin,True,0.8354,0.7917,0.7037,0.7451,0.9387,0.8545,0.9038,0.8785
7,8,NB,Gaussian Naive Bayes,Probabilistic,True,0.8228,0.7600,0.7037,0.7308,0.9003,0.8519,0.8846,0.8679
8,9,KNN,K-Nearest Neighbors,Instance-based,True,0.7468,0.7059,0.4444,0.5455,0.7393,0.7581,0.9038,0.8246


In [19]:
# Enforce structural ensemble baseline assertions
assert rfc.n_estimators == 300, "Error: Estimator count parameter does not match specified configuration (300)!"
assert rfc.random_state == 42, "Error: random_state configuration mismatch!"
assert len(rfc_predictions) == len(ycte), "Error: Output shape mismatch between targets and prediction vectors!"
assert np.isfinite(rfc_probabilities).all(), "Error: Predicted probabilities contain non-finite numbers!"

print("=" * 72)
print("SESSION 35 GITHUB DELIVERABLE COMPLETED SUCCESSFULLY")
print("=" * 72)

SESSION 35 GITHUB DELIVERABLE COMPLETED SUCCESSFULLY


## Session 36: Gradient Boosting and AdaBoost Classification
This section implements two sequential boosting ensembles: **Gradient Boosting** and **AdaBoost** classifiers. Boosting algorithms train weak learners sequentially, where each subsequent model focuses on correcting the residual errors or misclassifications made by the preceding trees. We evaluate whether sequential optimization yields superior at-risk detection relative to bagging ensembles.

In [20]:
# Session 36 imports and environment verification
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier
import numpy as np
import pandas as pd
from IPython.display import display

# Verify required splits and evaluation helpers exist in the active namespace
required_session36_objects = ["Xtr_f", "Xte_f", "yctr", "ycte", "evaluate_session33_classifier"]
missing_objects = [obj for obj in required_session36_objects if obj not in globals()]

if missing_objects:
    raise NameError(f"Prerequisite objects missing from prior sections: {missing_objects}. Please run preceding cells first.")

print("Session 36 prerequisites successfully verified. Datasets are ready for boosting ensemble training.")

Session 36 prerequisites successfully verified. Datasets are ready for boosting ensemble training.


In [21]:
# Initialize boosting estimators
boosting_estimators = [
    ("GradBoost", "Gradient Boosting Classifier", GradientBoostingClassifier(random_state=42)),
    ("AdaBoost", "AdaBoost Classifier", AdaBoostClassifier(random_state=42))
]

session36_result_rows = []

for model_code, full_model_name, estimator in boosting_estimators:
    # Fit model using raw training features
    estimator.fit(Xtr_f, yctr)
    
    # Generate class and probability predictions
    predictions = estimator.predict(Xte_f)
    probabilities = estimator.predict_proba(Xte_f)[:, 1]
    
    # Evaluate metrics using the project evaluation function
    metrics = evaluate_session33_classifier(y_true=ycte, y_pred=predictions, y_probability=probabilities)
    
    session36_result_rows.append({
        "Model": model_code,
        "Full Model Name": full_model_name,
        "Model Family": "Boosting",
        "Scaling_Used": False,
        **metrics
    })
    
    print(f"{model_code} Completed - F1: {metrics['f1']:.4f} | At-Risk F1: {metrics['at_risk_f1']:.4f}")

session36_results_df = pd.DataFrame(session36_result_rows)

GradBoost Completed - F1: 0.8000 | At-Risk F1: 0.8932
AdaBoost Completed - F1: 0.8519 | At-Risk F1: 0.9231


/usr/lib/python3/dist-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


In [22]:
# Path to cumulative metrics tracker
PROJECT_ROOT = Path.cwd().resolve()
for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (parent / ".git").exists():
        PROJECT_ROOT = parent
        break

cumulative_path = PROJECT_ROOT / "reports" / "tables" / "classification_model_comparison.csv"

if cumulative_path.exists():
    classification_table = pd.read_csv(cumulative_path)
    
    # Clear prior historical runs of GradBoost or AdaBoost rows to prevent duplicates
    if "Model" in classification_table.columns:
        classification_table = classification_table[~classification_table["Model"].isin(["GradBoost", "AdaBoost"])].copy()
        
    classification_table = pd.concat([classification_table, session36_results_df], ignore_index=True, sort=False)
else:
    classification_table = session36_results_df.copy()

# Sort the table globally by standard baseline F1
classification_table = classification_table.sort_values(by="f1", ascending=False).reset_index(drop=True)

if "Overall F1 Rank" in classification_table.columns:
    classification_table = classification_table.drop(columns=["Overall F1 Rank"])
classification_table.insert(0, "Overall F1 Rank", range(1, len(classification_table) + 1))

# Export updated tables
classification_table.to_csv(cumulative_path, index=False)
print(f"Cumulative tracking table written successfully to: {cumulative_path}")
display(classification_table.round(4))

Cumulative tracking table written successfully to: /home/nikhil/Desktop/VSCode/GSSRP/student-performance-prediction-ml/reports/tables/classification_model_comparison.csv


,Overall F1 Rank,Model,Full Model Name,Model Family,Scaling_Used,accuracy,precision,recall,f1,roc_auc,at_risk_precision,at_risk_recall,at_risk_f1
0,1,DTC,Decision Tree Classifier,Tree-based,False,0.9114,0.8571,0.8889,0.8727,0.9099,0.9412,0.9231,0.9320
1,2,LR,Logistic Regression,Linear,True,0.8987,0.8276,0.8889,0.8571,0.9736,0.8276,0.8889,0.8571
2,3,RFC,Random Forest Classifier,Ensemble,False,0.8987,0.8276,0.8889,0.8571,0.9605,0.9400,0.9038,0.9216
3,4,AdaBoost,AdaBoost Classifier,Boosting,False,0.8987,0.8519,0.8519,0.8519,0.9729,0.9231,0.9231,0.9231
4,5,MLPC,Multi-Layer Perceptron Classifier,Neural Network,True,0.8734,0.8696,0.7407,0.8000,0.9423,0.8750,0.9423,0.9074
5,6,GradBoost,Gradient Boosting Classifier,Boosting,False,0.8608,0.7857,0.8148,0.8000,0.9608,0.9020,0.8846,0.8932
6,7,SVM,Support Vector Machine,Maximum-margin,True,0.8354,0.7917,0.7037,0.7451,0.9387,0.8545,0.9038,0.8785
7,8,NB,Gaussian Naive Bayes,Probabilistic,True,0.8228,0.7600,0.7037,0.7308,0.9003,0.8519,0.8846,0.8679
8,9,KNN,K-Nearest Neighbors,Instance-based,True,0.7468,0.7059,0.4444,0.5455,0.7393,0.7581,0.9038,0.8246


In [23]:
# Enforce structural boosting parameter and target shape checks
assert len(session36_results_df) == 2, "Error: Result rows mismatch inside standalone boosting table."
assert set(session36_results_df["Model"]) == {"GradBoost", "AdaBoost"}, "Error: Incomplete boosting model identifiers logged."

# Verify data prediction vector dimensions match targets
assert len(predictions) == len(ycte), "Error: Output shape mismatch between targets and prediction vectors."

print("=" * 72)
print("SESSION 36 NOTEBOOK DELIVERABLE COMPLETED")
print("=" * 72)

SESSION 36 NOTEBOOK DELIVERABLE COMPLETED


## Session 37: Neural-Network (MLP) Classification
This section implements a neural-network classifier using a **Multi-Layer Perceptron (MLP)** architecture. Neural network classifiers are highly flexible function approximators capable of modeling intricate multi-dimensional boundaries, but they are highly sensitive to input feature scales. To ensure stable gradient descent weight updates and model convergence, we encapsulate `StandardScaler` and `MLPClassifier` inside a scikit-learn `Pipeline`. The network is configured with two hidden layers of 64 and 32 neurons respectively, running for a maximum of 1000 iterations.

In [24]:
# Session 37 imports and environment verification
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import numpy as np
import pandas as pd
from IPython.display import display

# Verify required splits and evaluation helpers exist in the active namespace
required_session37_objects = ["Xtr_f", "Xte_f", "yctr", "ycte", "evaluate_session33_classifier"]
missing_objects = [obj for obj in required_session37_objects if obj not in globals()]

if missing_objects:
    raise NameError(f"Prerequisite objects missing from prior sections: {missing_objects}. Please run preceding cells first.")

print("Session 37 prerequisites successfully verified. Datasets are ready for neural network training.")

Session 37 prerequisites successfully verified. Datasets are ready for neural network training.


In [25]:
# Initialize scaled MLPClassifier pipeline
mlpc_pipeline = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42)
)

# Fit the MLP pipeline on the binary training data
mlpc_pipeline.fit(Xtr_f, yctr)

# Generate class and probability predictions
mlpc_predictions = mlpc_pipeline.predict(Xte_f)
mlpc_probabilities = mlpc_pipeline.predict_proba(Xte_f)[:, 1]

# Evaluate metrics using the project evaluation function
mlpc_metrics = evaluate_session33_classifier(y_true=ycte, y_pred=mlpc_predictions, y_probability=mlpc_probabilities)

print("MLP Classifier Evaluation (Test Split):")
print("-" * 50)
for metric_name, metric_val in mlpc_metrics.items():
    print(f"{metric_name}: {metric_val:.4f}")

MLP Classifier Evaluation (Test Split):
--------------------------------------------------
accuracy: 0.8734
precision: 0.8696
recall: 0.7407
f1: 0.8000
roc_auc: 0.9423
at_risk_precision: 0.8750
at_risk_recall: 0.9423
at_risk_f1: 0.9074


In [26]:
# Path to cumulative metrics tracker
PROJECT_ROOT = Path.cwd().resolve()
for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (parent / ".git").exists():
        PROJECT_ROOT = parent
        break

cumulative_path = PROJECT_ROOT / "reports" / "tables" / "classification_model_comparison.csv"

# Structure the output tracking row matching the cumulative schema
mlpc_row = pd.DataFrame([{
    "Model": "MLPC",
    "Full Model Name": "Multi-Layer Perceptron Classifier",
    "Model Family": "Neural Network",
    "Scaling_Used": True,
    "accuracy": mlpc_metrics["accuracy"],
    "precision": mlpc_metrics["precision"],
    "recall": mlpc_metrics["recall"],
    "f1": mlpc_metrics["f1"],
    "roc_auc": mlpc_metrics["roc_auc"],
    "at_risk_precision": mlpc_metrics["at_risk_precision"],
    "at_risk_recall": mlpc_metrics["at_risk_recall"],
    "at_risk_f1": mlpc_metrics["at_risk_f1"]
}])

if cumulative_path.exists():
    classification_table = pd.read_csv(cumulative_path)
    
    # Clear prior historical runs of MLPC rows to prevent duplicates
    if "Model" in classification_table.columns:
        classification_table = classification_table[classification_table["Model"] != "MLPC"].copy()
        
    classification_table = pd.concat([classification_table, mlpc_row], ignore_index=True, sort=False)
else:
    classification_table = mlpc_row.copy()

# Sort the table globally by standard baseline F1
classification_table = classification_table.sort_values(by="f1", ascending=False).reset_index(drop=True)

if "Overall F1 Rank" in classification_table.columns:
    classification_table = classification_table.drop(columns=["Overall F1 Rank"])
classification_table.insert(0, "Overall F1 Rank", range(1, len(classification_table) + 1))

# Export updated tables
classification_table.to_csv(cumulative_path, index=False)
print(f"Cumulative tracking table written successfully to: {cumulative_path}")
display(classification_table.round(4))

Cumulative tracking table written successfully to: /home/nikhil/Desktop/VSCode/GSSRP/student-performance-prediction-ml/reports/tables/classification_model_comparison.csv


,Overall F1 Rank,Model,Full Model Name,Model Family,Scaling_Used,accuracy,precision,recall,f1,roc_auc,at_risk_precision,at_risk_recall,at_risk_f1
0,1,DTC,Decision Tree Classifier,Tree-based,False,0.9114,0.8571,0.8889,0.8727,0.9099,0.9412,0.9231,0.9320
1,2,LR,Logistic Regression,Linear,True,0.8987,0.8276,0.8889,0.8571,0.9736,0.8276,0.8889,0.8571
2,3,RFC,Random Forest Classifier,Ensemble,False,0.8987,0.8276,0.8889,0.8571,0.9605,0.9400,0.9038,0.9216
3,4,AdaBoost,AdaBoost Classifier,Boosting,False,0.8987,0.8519,0.8519,0.8519,0.9729,0.9231,0.9231,0.9231
4,5,GradBoost,Gradient Boosting Classifier,Boosting,False,0.8608,0.7857,0.8148,0.8000,0.9608,0.9020,0.8846,0.8932
5,6,MLPC,Multi-Layer Perceptron Classifier,Neural Network,True,0.8734,0.8696,0.7407,0.8000,0.9423,0.8750,0.9423,0.9074
6,7,SVM,Support Vector Machine,Maximum-margin,True,0.8354,0.7917,0.7037,0.7451,0.9387,0.8545,0.9038,0.8785
7,8,NB,Gaussian Naive Bayes,Probabilistic,True,0.8228,0.7600,0.7037,0.7308,0.9003,0.8519,0.8846,0.8679
8,9,KNN,K-Nearest Neighbors,Instance-based,True,0.7468,0.7059,0.4444,0.5455,0.7393,0.7581,0.9038,0.8246


In [27]:
# Extract steps to validate structural integrity
scaler_step = mlpc_pipeline.steps[0][1]
mlpc_step = mlpc_pipeline.steps[1][1]

# Enforce parameters and shape constraints
assert isinstance(scaler_step, StandardScaler), "Error: First pipeline step must be StandardScaler!"
assert isinstance(mlpc_step, MLPClassifier), "Error: Second pipeline step must be MLPClassifier!"
assert mlpc_step.hidden_layer_sizes == (64, 32), "Error: Hidden layer sizes must be set to (64, 32)!"
assert mlpc_step.max_iter == 1000, "Error: max_iter must be configured to 1000!"
assert mlpc_step.random_state == 42, "Error: random_state configuration mismatch!"
assert len(mlpc_predictions) == len(ycte), "Error: Prediction output vector shape mismatch!"

print("=" * 72)
print("SESSION 37 NOTEBOOK DELIVERABLE COMPLETED")
print("=" * 72)

SESSION 37 NOTEBOOK DELIVERABLE COMPLETED
